In [1]:
# 1: importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3


In [2]:
# 2: data importing and exploring

try: 
    df = pd.read_csv("../data/superstore.csv", encoding='latin-1')
except FileNotFoundError:
    print("File not found")


# df.head()
# df.isnull().sum()
df.dtypes
# df.describe()
# df.columns
# df.head()

Row ID             int64
Order ID          object
Order Date        object
Ship Date         object
Ship Mode         object
Customer ID       object
Customer Name     object
Segment           object
Country           object
City              object
State             object
Postal Code        int64
Region            object
Product ID        object
Category          object
Sub-Category      object
Product Name      object
Sales            float64
Quantity           int64
Discount         float64
Profit           float64
dtype: object

In [3]:
# 3: data cleaning
df.drop_duplicates(inplace=True)
df['Order Date'] = pd.to_datetime(
            df['Order Date'],
            format='%m/%d/%Y'
        )

df['Order Date'] = df['Order Date'].dt.strftime('%Y-%m-%d')

# print(df['Order Date'].dtype)

df['Ship Date'] = pd.to_datetime(
            df['Ship Date'],
            format='%m/%d/%Y'
    )

df['Ship Date'] = df['Ship Date'].dt.strftime('%Y-%m-%d')

# print(df['Ship Date'].dtype)

# df.dtypes

In [4]:
# SQLite connection
try:
    conn = sqlite3.connect(':memory:')
    print('Connection made')
    df.to_sql('superstore', conn, if_exists='replace', index=False)
    print('BD criada')

except:
    print('Impossivel conectar ao SQLite')



Connection made
BD criada


In [5]:
# Executes queries
# P.S.: run this section before running any query from the business questions

def execute(query):
    try:
        result = pd.read_sql_query(query, conn)
        return result
    except:
        print('Erro ao ler query')

In [ ]:
# 4: business questions

# 1. Which region generates the most total sales?
query = """
        SELECT Region, max(total_sales) AS "Total Sales"
        FROM ( 
            SELECT
                Region,
                ROUND(SUM(sales), 2) AS total_sales
            FROM superstore
            GROUP BY region
        ) AS maxi
    """

print(execute(query))


  Region  Total Sales
0   West    725457.82


In [ ]:
# 2. What are the top 5 products by total revenue?
query = """
        SELECT
            "Product Name",
            ROUND(SUM(profit), 2) AS Revenue
        FROM superstore
        GROUP BY "Product Name"
        ORDER BY revenue DESC
        LIMIT 5
    """

print(execute(query))

                                        Product Name   Revenue
0              Canon imageCLASS 2200 Advanced Copier  25199.93
1  Fellowes PB500 Electric Punch Plastic Comb Bin...   7753.04
2               Hewlett Packard LaserJet 3310 Copier   6983.88
3                 Canon PC1060 Personal Laser Copier   4570.93
4  HP Designjet T520 Inkjet Large Format Printer ...   4094.98


In [ ]:
# 3. What do monthly sales trends look like over time?
query = """
        SELECT
            substr(
                'JanFebMarAprMayJunJulAugSepOctNovDec',
                1 + 3*(strftime('%m', datetime("Order Date")) - 1),
                3
            ) AS Month,

            strftime('%Y', datetime("Order Date")) AS Year,
            ROUND(SUM(Sales), 2) AS "Month-Year Sales"
        FROM superstore
        GROUP BY Year, Month
        ORDER BY Year, strftime('%m', datetime("Order Date"))
    """

print(execute(query))

   Month  Year  Month-Year Sales
0    Jan  2014          14236.90
1    Feb  2014           4519.89
2    Mar  2014          55691.01
3    Apr  2014          28295.35
4    May  2014          23648.29
5    Jun  2014          34595.13
6    Jul  2014          33946.39
7    Aug  2014          27909.47
8    Sep  2014          81777.35
9    Oct  2014          31453.39
10   Nov  2014          78628.72
11   Dec  2014          69545.62
12   Jan  2015          18174.08
13   Feb  2015          11951.41
14   Mar  2015          38726.25
15   Apr  2015          34195.21
16   May  2015          30131.69
17   Jun  2015          24797.29
18   Jul  2015          28765.33
19   Aug  2015          36898.33
20   Sep  2015          64595.92
21   Oct  2015          31404.92
22   Nov  2015          75972.56
23   Dec  2015          74919.52
24   Jan  2016          18542.49
25   Feb  2016          22978.82
26   Mar  2016          51715.88
27   Apr  2016          38750.04
28   May  2016          56987.73
29   Jun  

In [ ]:
# 4. Which category has the best profit margin?
query = """
        SELECT
            Category,
            MAX(Profit_Margin) AS "Best Profit Margin (%)"
        FROM (
            SELECT
                Category,
                ROUND((SUM(Profit) / SUM(Sales)) * 100, 2) AS Profit_Margin
            FROM superstore
            GROUP BY Category
        ) AS PF
    """

print(execute(query))

     Category  Best Profit Margin (%)
0  Technology                    17.4


In [ ]:
# 5. Which sub-categories are losing money?
query = """
        SELECT
            "Sub-Category",
            ROUND(SUM(Profit), 2) AS "Total Profit"
        FROM superstore
        GROUP BY "Sub-Category"
        HAVING "Total Profit" < 0
        ORDER BY "Sub-Category";
    """

print(execute(query))

  Sub-Category  Total Profit
0    Bookcases      -3472.56
1     Supplies      -1189.10
2       Tables     -17725.48


In [ ]:
# 6. Does offering a higher discount actually hurt profit?
query = """
        SELECT
            CASE
                WHEN Discount < 0.1 THEN '0-10%'
                WHEN Discount < 0.2 THEN '10-20%'
                WHEN Discount < 0.3 THEN '20-30%'
                WHEN Discount < 0.4 THEN '30-40%'
                ELSE '50+%'
            END AS "Discount Range",
            ROUND(AVG(Profit), 2) AS "Profit Average",
            Count(*) AS "Number of Orders"
        FROM superstore
        GROUP BY "Discount Range"
    """

print(execute(query))

  Discount Range  Profit Average  Number of Orders
0          0-10%           66.90              4798
1         10-20%           71.56               146
2         20-30%           24.70              3657
3         30-40%          -50.24               254
4           50+%         -107.65              1139


In [ ]:
# 7. Which customer segment (Consumer, Corporate, Home Office) is most valuable?
query = """
        SELECT
            Segment,
            ROUND(SUM(Profit), 2) AS "Total Profit",
            ROUND(AVG(Profit), 2) AS "Average Profit",
            COUNT(*) AS "Number of Orders"
        FROM superstore
        GROUP BY Segment
        ORDER BY "Total Profit" DESC
    """

print(execute(query))

       Segment  Total Profit  Average Profit  Number of Orders
0     Consumer     134119.21           25.84              5191
1    Corporate      91979.13           30.46              3020
2  Home Office      60298.68           33.82              1783


In [ ]:
# 8. What's the average order value per region?

query = """
        SELECT
            Region,
            ROUND(SUM(Sales) / COUNT(*), 2) AS "Average Order Value"
        FROM superstore
        GROUP BY Region
        ORDER BY Region
    """

print(execute(query))

    Region  Average Order Value
0  Central               215.77
1     East               238.34
2    South               241.80
3     West               226.49


In [ ]:
# 9. Which shipping mode is used most, and does it correlate with order size?

query = """
        SELECT
            "Ship Mode",
            COUNT(*) AS "Number of Orders",
            ROUND(AVG(Quantity), 2) AS "Order Size Average"
        FROM superstore
        GROUP BY "Ship Mode"
        ORDER BY "Number of Orders" DESC
    """

print(execute(query))

        Ship Mode  Number of Orders  Order Size Average
0  Standard Class              5968                3.82
1    Second Class              1945                3.82
2     First Class              1538                3.70
3        Same Day               543                3.61


In [67]:
# 10. Which state has the most orders but lowest profit? (hidden underperformer)

query = """
        SELECT
            "State",
            COUNT(*) AS "Number of Orders",
            ROUND(SUM(Profit), 2) AS "Total Profit"
        FROM superstore
        GROUP BY State
        ORDER BY "Number of Orders" DESC, "Total Profit" ASC
    """

print(execute(query))

                   State  Number of Orders  Total Profit
0             California              2001      76381.39
1               New York              1128      74038.55
2                  Texas               985     -25729.36
3           Pennsylvania               587     -15559.96
4             Washington               506      33402.65
5               Illinois               492     -12607.89
6                   Ohio               469     -16971.38
7                Florida               383      -3399.30
8               Michigan               255      24463.19
9         North Carolina               249      -7490.91
10               Arizona               224      -3427.92
11              Virginia               224      18597.95
12               Georgia               184      16250.04
13             Tennessee               183      -5341.69
14              Colorado               182      -6527.86
15               Indiana               149      18382.94
16              Kentucky       